# 02 — Clean

Loads the raw pull from `data/crimes_raw.parquet`, parses dates and derives time features, validates coordinates and community areas, groups the 33 raw crime types into 4 categories, and saves the trimmed result to `data/crimes_clean.parquet`.

In [ ]:
import pandas as pd

df = pd.read_parquet("data/crimes_raw.parquet")
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.head()

## Schema

In [ ]:
df.dtypes

## Time features

Parse the date string into a real datetime and derive the year, month, hour, and day-of-week columns the temporal analysis needs.

In [ ]:
# parse the date string into a real datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# derive time features for the temporal analysis
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["hour"] = df["date"].dt.hour
df["dayofweek"] = df["date"].dt.day_name()       # "Monday", "Tuesday", ...
df["month_name"] = df["date"].dt.month_name()    # "January", ... (handy for viz)

# check it worked
print(df["date"].dtype)
print(df[["date", "year", "month", "hour", "dayofweek"]].head())

## Coordinate validation

Coerce latitude/longitude to numeric and drop rows where either is missing — those can't be placed on the map.

In [ ]:
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

before = len(df)
df = df.dropna(subset=["latitude", "longitude"])
after = len(df)
print(f"Dropped {before - after:,} rows missing coordinates")
print(f"{after:,} rows remain")

## Community area validation

Chicago has 77 community areas; drop rows missing one and confirm the count is close to that.

In [ ]:
df["community_area"] = pd.to_numeric(df["community_area"], errors="coerce")

before = len(df)
df = df.dropna(subset=["community_area"])
df["community_area"] = df["community_area"].astype(int)
print(f"Dropped {before - len(df):,} rows missing community area")
print(f"{df['community_area'].nunique()} unique community areas (should be ~77)")

## Crime type distribution

Check what the raw `primary_type` values actually look like before deciding how to group them.

In [ ]:
print(df["primary_type"].nunique(), "unique crime types\n")
print(df["primary_type"].value_counts())

## Categorize crime types

Group the 33 raw crime types into 4 categories (Violent, Property, Quality-of-Life, Other) for the dashboard filters. The unmapped-type check below is a safety net — it should print nothing on a clean run.

In [ ]:
crime_category_map = {
    # ---- Violent ----
    "BATTERY": "Violent",
    "ASSAULT": "Violent",
    "ROBBERY": "Violent",
    "HOMICIDE": "Violent",
    "CRIMINAL SEXUAL ASSAULT": "Violent",
    "SEX OFFENSE": "Violent",
    "CRIM SEXUAL ASSAULT": "Violent",      # older label, just in case
    "KIDNAPPING": "Violent",
    "HUMAN TRAFFICKING": "Violent",
    "INTIMIDATION": "Violent",
    "STALKING": "Violent",
    "OFFENSE INVOLVING CHILDREN": "Violent",

    # ---- Property ----
    "THEFT": "Property",
    "CRIMINAL DAMAGE": "Property",
    "MOTOR VEHICLE THEFT": "Property",
    "BURGLARY": "Property",
    "ARSON": "Property",
    "CRIMINAL TRESPASS": "Property",

    # ---- Quality-of-Life ----
    "NARCOTICS": "Quality-of-Life",
    "WEAPONS VIOLATION": "Quality-of-Life",
    "PUBLIC PEACE VIOLATION": "Quality-of-Life",
    "PROSTITUTION": "Quality-of-Life",
    "LIQUOR LAW VIOLATION": "Quality-of-Life",
    "GAMBLING": "Quality-of-Life",
    "INTERFERENCE WITH PUBLIC OFFICER": "Quality-of-Life",
    "CONCEALED CARRY LICENSE VIOLATION": "Quality-of-Life",
    "OTHER NARCOTIC VIOLATION": "Quality-of-Life",
    "PUBLIC INDECENCY": "Quality-of-Life",
    "OBSCENITY": "Quality-of-Life",

    # ---- Other ----
    "DECEPTIVE PRACTICE": "Other",
    "OTHER OFFENSE": "Other",
    "NON-CRIMINAL": "Other",
    "NON - CRIMINAL": "Other",
    "NON-CRIMINAL (SUBJECT SPECIFIED)": "Other",
    "RITUALISM": "Other",
}

df["crime_category"] = df["primary_type"].map(crime_category_map)

# safety net: anything not in the map shows up here
unmapped = df[df["crime_category"].isna()]["primary_type"].value_counts()
if len(unmapped) > 0:
    print("⚠️ Unmapped types (need to add these):")
    print(unmapped)
else:
    print("✅ All crime types mapped")

print("\nCategory breakdown:")
print(df["crime_category"].value_counts())

## Save

Keep only the columns the aggregation stage needs and write `data/crimes_clean.parquet`.

In [ ]:
keep_cols = [
    "id", "date", "year", "month", "month_name", "hour", "dayofweek",
    "primary_type", "crime_category", "description",
    "arrest", "domestic",
    "community_area", "district", "ward",
    "latitude", "longitude", "location_description"
]
clean = df[keep_cols].copy()

print(f"Clean dataset: {len(clean):,} rows, {clean.shape[1]} columns")
clean.head()

In [ ]:
import os
clean.to_parquet("data/crimes_clean.parquet", index=False)

size_mb = os.path.getsize("data/crimes_clean.parquet") / 1e6
print(f"Saved data/crimes_clean.parquet ({size_mb:.1f} MB)")
print(os.listdir("data"))